# MRI (T1) -> DTI U-Net Training Pipeline (Kaggle) — 3D Volume/Patch U-Net — Log-Euclidean SPD Tensor Synthesis

This is the single notebook we're continuing with going forward (2D and FA+MD-only variants retired).

Trains a 3D U-Net that predicts the diffusion tensor in **log-Euclidean space**: the network
outputs 6 unconstrained real values per voxel, which are mapped through a symmetric matrix
exponential to guarantee the result is always a physically valid (SPD — symmetric
positive-definite) diffusion tensor **by construction**, not by post-hoc correction. This
matches the architecture described in the project's Review I slides and is the same approach
used in the closest prior work (Manifold-Aware Synthesis of High-Resolution Diffusion from
Structural Imaging, 2022).

FA and MD are *derived* from the predicted tensor's eigenvalues at evaluation/visualization
time — they are not separately predicted channels.

**Groups used:** ADNI-AD, ADNI-MCI, ADNI-CN, HCP.

**Before running:**
- Settings panel (right sidebar) -> **Accelerator: GPU T4 x2** or **GPU P100**
- Settings panel -> **Internet: On** (only needed for the pip install cell)
- Add your `processed_hcp` and `processed_adni` outputs as Kaggle Datasets (Add Input, right sidebar) if they aren't already in this session's `/kaggle/working/`

**Resilience built in:**
- Auto-discovery skips corrupted/incomplete subject folders instead of crashing
- Volumes are cached to disk as memory-mapped files (Step 3) — avoids the RAM blowup that
  crashed earlier runs
- Checkpoints every epoch to `/kaggle/working/checkpoints/` — if your session disconnects, just
  re-run the notebook top to bottom; it auto-resumes from the last saved epoch
- NaN/Inf losses and CUDA OOM on a single batch are caught and skipped, not fatal
- Early stopping on validation loss

**All 5 metrics from the Review I slides are covered**, split across three stages:
- Step 9: FA/MD MSE, SSIM, cosine similarity of principal diffusion direction — each broken
  down by group (AD/MCI/CN/HCP) and pooled
- Step 11: Bidirectional Synthesis Check (MRI -> synthetic DTI -> MRI', compared to the original)
- Step 12: Diagnostic Validity Check (AD/MCI/CN classifier, real DTI vs. synthetic DTI)
- Step 13: Tractography overlap (Dice) — a custom, simplified deterministic tracker, since this
  pipeline never has raw DWI + a gradient table for a full `dipy` pipeline; see Step 13's
  markdown for exactly what that means for how to interpret the number


## Step 0 - GPU check

In [ ]:
!nvidia-smi

## Install & import dependencies

In [ ]:
!pip install -q --no-cache-dir nibabel scikit-image tqdm
import torch
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
import os, re, json, time, random, shutil, warnings, traceback
import numpy as np
import nibabel as nib
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy.ndimage import zoom as nd_zoom
from skimage.transform import resize as sk_resize
from skimage.metrics import structural_similarity as sk_ssim
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)


## Step 1 - Auto-discover processed subjects

Walks `/kaggle/input/` and `/kaggle/working/` looking for:
- `.../processed_hcp/<subject_id>/` (needs `T1_brain`, `FA`, `MD` and `tensor` — `.nii` or `.nii.gz`, both handled)
- `.../processed_adni/{AD,MCI,CN}/<subject_id>/` (needs `T1_brain`, `FA`, `MD` and `tensor_lower_triangular` — `.nii` or `.nii.gz`, both handled)

Note the tensor filename differs between the two pipelines (`tensor` for HCP vs.
`tensor_lower_triangular` for ADNI) — handled below. All three ADNI diagnostic groups
(AD, MCI, CN) are discovered.
Any subject missing a required file, or whose files fail to open, is skipped and logged —
it will not crash the run.


In [ ]:
REQUIRE_TENSOR = True

HCP_TENSOR_BASE = "tensor"
ADNI_TENSOR_BASE = "tensor_lower_triangular"

SEARCH_ROOTS = ["/kaggle/input", "/kaggle/working"]


def _resolve_ext(subj_dir, base_name):
    """Preprocessing output has been seen as both .nii and .nii.gz — try both,
    .nii.gz first since that's what the pipeline docs describe, .nii is what's
    actually on disk for this project. Returns the real path, or None."""
    for ext in (".nii.gz", ".nii"):
        candidate = os.path.join(subj_dir, base_name + ext)
        if os.path.exists(candidate):
            return candidate
    return None


def _file_ok(path):
    """Cheap validity check: file exists, opens, has a non-degenerate shape."""
    if path is None or not os.path.exists(path):
        return False
    try:
        img = nib.load(path)
        shape = img.shape
        if len(shape) < 3 or any(s <= 1 for s in shape[:3]):
            return False
        return True
    except Exception:
        return False


import re

def _find_processed_dirs(root_name):
    """Find every directory matching `root_name` under the search roots, tolerating
    Kaggle's automatic renaming when multiple attached datasets contain a folder with the
    same name (e.g. a second `processed_adni` upload becomes `processed_adni (1)`, and a
    second `processed_hcp` upload becomes `processed_hcp 2`). An exact-name match alone
    misses these renamed folders entirely — this happened in practice, silently dropping
    every CN subject from discovery even though the files were present on disk."""
    pattern = re.compile(rf"^{re.escape(root_name)}(\s*\(\d+\))?(\s+\d+)?$")
    hits = []
    for base in SEARCH_ROOTS:
        if not os.path.isdir(base):
            continue
        for root, dirs, files in os.walk(base):
            for d in dirs:
                if pattern.match(d):
                    hits.append(os.path.join(root, d))
    if hits:
        print(f"  {root_name}: matched folder(s) -> {[os.path.basename(h) for h in hits]}")
    return hits


def discover_subjects():
    manifest = []
    skipped = []

    # --- HCP ---
    for hcp_root in _find_processed_dirs("processed_hcp"):
        for subj in sorted(os.listdir(hcp_root)):
            subj_dir = os.path.join(hcp_root, subj)
            if not os.path.isdir(subj_dir):
                continue
            t1 = _resolve_ext(subj_dir, "T1_brain")
            fa = _resolve_ext(subj_dir, "FA")
            md_ = _resolve_ext(subj_dir, "MD")
            tensor = _resolve_ext(subj_dir, HCP_TENSOR_BASE)
            required = [t1, fa, md_] + ([tensor] if REQUIRE_TENSOR else [])
            if not all(_file_ok(p) for p in required):
                skipped.append((f"HCP:{subj}", "missing/corrupt required file(s)"))
                continue
            manifest.append(dict(subject_id=subj, group="HCP", t1=t1, fa=fa, md=md_,
                                  tensor=tensor if REQUIRE_TENSOR else None))

    # --- ADNI (AD, MCI only) ---
    for adni_root in _find_processed_dirs("processed_adni"):
        for cat in ("AD", "MCI", "CN"):
            cat_dir = os.path.join(adni_root, cat)
            if not os.path.isdir(cat_dir):
                continue
            for subj in sorted(os.listdir(cat_dir)):
                subj_dir = os.path.join(cat_dir, subj)
                if not os.path.isdir(subj_dir):
                    continue
                t1 = _resolve_ext(subj_dir, "T1_brain")
                fa = _resolve_ext(subj_dir, "FA")
                md_ = _resolve_ext(subj_dir, "MD")
                tensor = _resolve_ext(subj_dir, ADNI_TENSOR_BASE)
                required = [t1, fa, md_] + ([tensor] if REQUIRE_TENSOR else [])
                if not all(_file_ok(p) for p in required):
                    skipped.append((f"{cat}:{subj}", "missing/corrupt required file(s)"))
                    continue
                manifest.append(dict(subject_id=subj, group=cat, t1=t1, fa=fa, md=md_,
                                      tensor=tensor if REQUIRE_TENSOR else None))

    # de-duplicate by (group, subject_id) in case the same processed folder is mounted twice
    seen = set()
    deduped = []
    for m in manifest:
        key = (m["group"], m["subject_id"])
        if key in seen:
            continue
        seen.add(key)
        deduped.append(m)

    return deduped, skipped


manifest, skipped = discover_subjects()

from collections import Counter
counts = Counter(m["group"] for m in manifest)
print("Discovered usable subjects:")
for g in ("AD", "MCI", "CN", "HCP"):
    print(f"  {g}: {counts.get(g, 0)}")
print(f"Total: {len(manifest)}")

if skipped:
    print(f"\nSkipped {len(skipped)} subject(s) (missing/corrupt files):")
    for name, reason in skipped[:20]:
        print(f"  - {name}: {reason}")
    if len(skipped) > 20:
        print(f"  ... and {len(skipped)-20} more")

if len(manifest) == 0:
    raise RuntimeError(
        "No usable subjects found. Check that processed_hcp / processed_adni are attached "
        "as Kaggle Inputs or present in /kaggle/working/, and that folder names match the "
        "preprocessing notebooks' output structure."
    )


## Step 2 - Stratified subject-level train/val/test split

Splits within each group (AD, MCI, HCP) separately at a ~75/15/10 ratio, then merges —
this avoids an unbalanced val/test set and never puts slices/patches from the same subject
in two different splits.


In [ ]:
# ===================== PATCH P1 — replaces Step 2 ======================================
# Uses the frozen split from notebook 02 (split.csv) instead of re-splitting. Re-splitting
# after adding subjects would reshuffle every group and change the paper's 43-subject test set.
import glob
import pandas as pd

SPLIT_CSV = None   # None = find the single split.csv attached under /kaggle/input
if SPLIT_CSV is None:
    _hits = glob.glob("/kaggle/input/**/split.csv", recursive=True)
    if len(_hits) != 1:
        raise RuntimeError(f"Attach the output of notebook 02 (exactly one split.csv). Found: {_hits}")
    SPLIT_CSV = _hits[0]
split_df = pd.read_csv(SPLIT_CSV, dtype={"subject_id": str})
_assign = {(g, s): sp for g, s, sp in zip(split_df.group, split_df.subject_id, split_df.split)}

train_manifest, val_manifest, test_manifest, _unassigned = [], [], [], []
for m in manifest:
    sp = _assign.get((m["group"], str(m["subject_id"])))
    {"train": train_manifest, "val": val_manifest, "test": test_manifest}.get(sp, _unassigned).append(m)

random.Random(SEED).shuffle(train_manifest)
for g in ("AD", "MCI", "CN", "HCP"):
    n = [sum(e["group"] == g for e in lst) for lst in (train_manifest, val_manifest, test_manifest)]
    print(f"{g}: train={n[0]} val={n[1]} test={n[2]}")
print(f"\nTOTAL -> train={len(train_manifest)} val={len(val_manifest)} test={len(test_manifest)}   (split file: {SPLIT_CSV})")
if _unassigned:
    print(f"[WARN] {len(_unassigned)} discovered subject(s) are not in split.csv and are NOT used "
          f"(re-run notebook 02 if they should be): {[(e['group'], e['subject_id']) for e in _unassigned[:5]]}")
_n_test = int((split_df.split == "test").sum())
if len(test_manifest) != _n_test:
    raise RuntimeError(f"Only {len(test_manifest)}/{_n_test} frozen test subjects were found. Attach all old datasets.")

## Step 3 - Volume loading, normalization & caching

Loads each subject's T1 / FA / MD + full tensor volumes once, resamples them to a common
canonical shape, normalizes them, and caches the result **to disk** (`/kaggle/working/_volume_cache/`)
as memory-mapped `.npy` files rather than holding everything in RAM. This matters at your
subject count: holding ~150+ subjects' worth of resampled volumes as plain in-memory arrays
can run into tens of GB and get the kernel OOM-killed. Memory-mapped file pages are
reclaimable by the OS under memory pressure, so this stays safe regardless of dataset size —
only the small slice/patch actually being read at any moment is resident in RAM. As a bonus,
the cache survives a session restart, so re-running after a disconnect skips re-resampling
subjects that were already processed.

A subject that fails to load at this stage (corrupt NIfTI, unreadable affine, shape mismatch
that can't be resolved) is dropped from its split with a warning instead of crashing the run.


In [ ]:
CANONICAL_SHAPE = (96, 112, 96)  # (H, W, D) every volume is resampled to before use
TARGET_VOXEL_MM = 2.0  # isotropic physical voxel size every volume is resampled to
MD_SCALE = 3e-3       # typical max MD in mm^2/s; used to squash MD into ~[0,1]
TENSOR_SCALE = 3e-3   # rough scale for tensor lower-triangular components

CACHE_DIR = "/kaggle/working/_volume_cache"
os.makedirs(CACHE_DIR, exist_ok=True)


def _resample(volume, src_zooms, target_shape, target_voxel_mm=TARGET_VOXEL_MM, order=1):
    """Resamples using each volume's actual physical voxel spacing (mm, from its NIfTI
    header), NOT raw array-shape ratio. This distinction matters a great deal when pooling
    HCP (1.25mm native spacing) and ADNI (~2-2.5mm, and variable native matrix size even
    within ADNI): resampling by array-shape ratio alone ignores physical voxel size
    entirely, so a subject with a larger native matrix gets crushed disproportionately
    relative to one with a smaller native matrix, even if their real-world head size is
    the same. That makes voxel (i,j,k) correspond to a different anatomical location in
    different subjects, which silently breaks the entire premise of a voxel-wise loss.
    Resampling to a fixed physical voxel size first, then padding/cropping to the
    canonical array shape, keeps anatomy consistently sized and centered across subjects
    regardless of source dataset or native resolution."""
    zoom_factors = [sz / target_voxel_mm for sz in src_zooms]
    out = nd_zoom(volume, zoom_factors, order=order)
    out = _pad_or_crop(out, target_shape)
    return out


def _pad_or_crop(vol, target_shape):
    pads = []
    slices = []
    for cur, tgt in zip(vol.shape, target_shape):
        if cur < tgt:
            before = (tgt - cur) // 2
            after = tgt - cur - before
            pads.append((before, after))
            slices.append(slice(0, cur))
        else:
            pads.append((0, 0))
            start = (cur - tgt) // 2
            slices.append(slice(start, start + tgt))
    vol = vol[tuple(slices)]
    if any(p != (0, 0) for p in pads):
        vol = np.pad(vol, pads, mode="constant", constant_values=0)
    return vol


def _load_nifti(path):
    """Returns (data, zooms) — zooms is the (dx, dy, dz) physical voxel spacing in mm
    from the NIfTI header, required for spacing-aware resampling above."""
    img = nib.load(path)
    data = img.get_fdata(dtype=np.float32)
    zooms = img.header.get_zooms()[:3]
    return data, zooms


def _clean(arr):
    return np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)


def _cache_paths(entry):
    tag = f"{entry['group']}_{entry['subject_id']}"
    paths = dict(
        t1=os.path.join(CACHE_DIR, f"{tag}_t1.npy"),
        fa=os.path.join(CACHE_DIR, f"{tag}_fa.npy"),
        md=os.path.join(CACHE_DIR, f"{tag}_md.npy"),
    )
    if entry.get("tensor") is not None:
        paths["tensor"] = os.path.join(CACHE_DIR, f"{tag}_tensor.npy")
    return paths


def _load_cached(paths):
    """Memory-maps existing .npy cache files. Raises if any are missing/corrupt so the
    caller can fall back to rebuilding them."""
    out = {}
    for name, path in paths.items():
        out[name] = np.load(path, mmap_mode="r")
    return out


def load_subject_volumes(entry):
    """Returns a dict of memory-mapped, resampled, normalized arrays, or None if loading
    fails. Resampled volumes are written to disk as float16 .npy files on first access and
    memory-mapped on every subsequent access, so RAM usage stays flat no matter how many
    subjects are in the dataset."""
    paths = _cache_paths(entry)

    if all(os.path.exists(p) for p in paths.values()):
        try:
            return _load_cached(paths)
        except Exception:
            pass  # cache is corrupt/incomplete — fall through and rebuild it

    try:
        t1, t1_zooms = _load_nifti(entry["t1"])
        fa, fa_zooms = _load_nifti(entry["fa"])
        md_raw, md_zooms = _load_nifti(entry["md"])
        t1, fa, md_raw = _clean(t1), _clean(fa), _clean(md_raw)

        t1 = _resample(t1, t1_zooms, CANONICAL_SHAPE, order=1)
        fa = _resample(fa, fa_zooms, CANONICAL_SHAPE, order=1)
        md_raw = _resample(md_raw, md_zooms, CANONICAL_SHAPE, order=1)

        # per-volume min-max normalize T1 (skull-stripped, so background is ~0)
        t1_max = np.percentile(t1, 99.5)
        t1 = np.clip(t1 / (t1_max + 1e-8), 0, 1)

        fa = np.clip(fa, 0, 1)  # FA is already bounded [0,1] by definition
        md_norm = np.clip(md_raw / MD_SCALE, 0, 1)

        np.save(paths["t1"], t1.astype(np.float16))
        np.save(paths["fa"], fa.astype(np.float16))
        np.save(paths["md"], md_norm.astype(np.float16))
        del t1, fa, md_raw, md_norm

        if entry.get("tensor") is not None:
            tensor, tensor_zooms = _load_nifti(entry["tensor"])  # (H, W, D, 6) lower-triangular
            tensor = _clean(tensor)
            if tensor.ndim != 4 or tensor.shape[-1] != 6:
                raise ValueError(f"unexpected tensor shape {tensor.shape}")
            tensor_r = np.stack(
                [_resample(tensor[..., c], tensor_zooms, CANONICAL_SHAPE, order=1) for c in range(6)],
                axis=-1,
            )
            # Scale only — no per-component clipping. Clipping each of the 6
            # lower-triangular entries independently can destroy the tensor's
            # positive-definiteness, which breaks the log-Euclidean math in Step 6/7
            # (matrix log requires positive eigenvalues). A uniform positive scalar
            # multiply is safe: it preserves SPD-ness exactly. Any remaining edge
            # cases (near-zero/negative eigenvalues from noise, background voxels)
            # are handled with an eigenvalue floor downstream, not by clipping here.
            tensor_norm = (tensor_r / TENSOR_SCALE).astype(np.float32)
            np.save(paths["tensor"], tensor_norm.astype(np.float16))
            del tensor, tensor_r, tensor_norm

        return _load_cached(paths)

    except Exception as e:
        print(f"  [WARN] failed to load {entry['group']}:{entry['subject_id']} -> {e}")
        return None


def filter_loadable(entries, split_name):
    """Eagerly validates every subject in a split once, dropping failures with a log line."""
    good = []
    for e in entries:
        if load_subject_volumes(e) is not None:
            good.append(e)
    dropped = len(entries) - len(good)
    if dropped:
        print(f"  {split_name}: dropped {dropped} subject(s) that failed to load")
    return good
# ===================== PATCH P2 — optional affine-aware harmonisation ==================
# Set to True if notebook 02's alignment QC says so. Uses a separate cache folder, so the
# two methods can never mix. Every variant you compare must use the SAME setting.
USE_AFFINE_HARMONIZATION = True    # notebook 02 QC: ADNI T1/tensor misaligned under the old method

# ---------------------------------------------------------------------------------------
# Affine-aware harmonisation (drop-in alternative to Step 3's zoom-only resampling)
#
# Step 3 resamples every volume with scipy zoom + centre pad/crop, using only voxel SIZE.
# It ignores each file's orientation and position (the NIfTI affine). If a subject's T1 is
# stored in a different orientation or field of view than its diffusion data (common for
# ADNI: sagittal/coronal MPRAGE vs axial DTI), the T1 and the tensor end up misaligned in
# the canonical grid, and the network is asked to learn from mismatched voxels.
#
# This version resamples T1, FA, MD and the tensor into ONE shared world-space grid
# (RAS axes, TARGET_VOXEL_MM spacing, CANONICAL_SHAPE, centred on the diffusion brain), and
# rotates the tensor frame to match. Normalisation is identical to Step 3.
# Needs from Step 3: CANONICAL_SHAPE, TARGET_VOXEL_MM, MD_SCALE, TENSOR_SCALE, _clean.
# ---------------------------------------------------------------------------------------
from scipy.ndimage import affine_transform as _affine_transform

_PAIRS = [(0, 0), (0, 1), (1, 1), (0, 2), (1, 2), (2, 2)]   # dipy order: Dxx, Dxy, Dyy, Dxz, Dyz, Dzz
FSL_BVEC_X_FLIP = True   # dipy fits tensors in the bvec frame; FSL-style bvecs flip x when det(affine) > 0
FLIP_Y_SITES = ("006_",)  # ADNI sites whose gradient y-axis is flipped (found by notebook 02 QC): negate Dxy, Dyz


def _six_to_mat_np(t6):
    m = np.zeros(t6.shape[:-1] + (3, 3), np.float32)
    for k, (i, j) in enumerate(_PAIRS):
        m[..., i, j] = t6[..., k]
        m[..., j, i] = t6[..., k]
    return m


def _mat_to_six_np(m):
    return np.stack([m[..., i, j] for i, j in _PAIRS], -1)


def _resample_world(vol, src_aff, tgt_aff, order=1):
    M = np.linalg.inv(src_aff) @ tgt_aff                      # target voxel -> source voxel
    return _affine_transform(vol, M[:3, :3], offset=M[:3, 3], output_shape=CANONICAL_SHAPE,
                             order=order, mode="constant", cval=0.0)


def _tensor_frame_rotation(src_aff, tgt_aff):
    A = np.linalg.inv(tgt_aff[:3, :3]) @ src_aff[:3, :3]
    U, _, Vt = np.linalg.svd(A)
    R = U @ Vt
    if FSL_BVEC_X_FLIP and np.linalg.det(src_aff[:3, :3]) > 0:
        R = R @ np.diag([-1.0, 1.0, 1.0])
    return R.astype(np.float32)


def _usable_affine(img, fallback):
    """Some writers save the tensor with an identity affine; fall back to the FA/MD affine."""
    a = img.affine
    if fallback is not None and np.allclose(a[:3, :3], np.eye(3)) and not np.allclose(fallback[:3, :3], np.eye(3)):
        return fallback
    return a


def harmonize_affine_arrays(entry):
    """Returns t1, fa, md_norm, tensor_norm(H,W,D,6 or None) on the canonical grid, float32."""
    md_img = nib.load(entry["md"])
    md_raw = _clean(md_img.get_fdata(dtype=np.float32))
    com = np.array(np.nonzero(md_raw > 0)).mean(1)
    centre = (md_img.affine @ np.r_[com, 1.0])[:3]
    tgt = np.diag([TARGET_VOXEL_MM] * 3 + [1.0])
    tgt[:3, 3] = centre - TARGET_VOXEL_MM * (np.array(CANONICAL_SHAPE) - 1) / 2.0

    t1_img = nib.load(entry["t1"])
    t1 = _resample_world(_clean(t1_img.get_fdata(dtype=np.float32)), t1_img.affine, tgt)
    fa_img = nib.load(entry["fa"])
    fa = _resample_world(_clean(fa_img.get_fdata(dtype=np.float32)), fa_img.affine, tgt)
    md_r = _resample_world(md_raw, md_img.affine, tgt)

    t1_max = np.percentile(t1, 99.5)                       # same normalisation as Step 3
    t1 = np.clip(t1 / (t1_max + 1e-8), 0, 1)
    fa = np.clip(fa, 0, 1)
    md_norm = np.clip(md_r / MD_SCALE, 0, 1)

    tensor_norm = None
    if entry.get("tensor") is not None:
        tn_img = nib.load(entry["tensor"])
        tn = _clean(tn_img.get_fdata(dtype=np.float32))
        if tn.ndim == 5:
            tn = tn[..., 0, :]
        if tn.ndim != 4 or tn.shape[-1] != 6:
            raise ValueError(f"unexpected tensor shape {tn.shape}")
        taff = _usable_affine(tn_img, md_img.affine)
        comps = np.stack([_resample_world(tn[..., c], taff, tgt) for c in range(6)], -1)
        R = _tensor_frame_rotation(taff, tgt)
        comps = _mat_to_six_np(R @ _six_to_mat_np(comps) @ R.T)
        if str(entry.get("subject_id", "")).startswith(FLIP_Y_SITES):
            comps[..., [1, 4]] *= -1
        tensor_norm = (comps / TENSOR_SCALE).astype(np.float32)
    return t1.astype(np.float32), fa.astype(np.float32), md_norm.astype(np.float32), tensor_norm


# ------------------------------- QC helpers -------------------------------
def qc_mask_dice(t1, md_norm, t1_thr=0.05, md_thr=0.01):
    """Overlap of the T1 brain and the diffusion brain on the canonical grid (1.0 = perfect)."""
    a, b = t1 > t1_thr, md_norm > md_thr
    s = a.sum() + b.sum()
    return float(2 * (a & b).sum() / s) if s else float("nan")


def qc_offdiag_agreement(tensors, md_list, md_thr=0.01):
    """For each subject: correlation of its Dxy, Dxz, Dyz maps with the mean of all OTHER subjects,
    inside the shared brain. Consistent frames -> clearly positive; misaligned -> ~0;
    a flipped axis -> two of the three strongly negative."""
    offd = [1, 3, 4]   # Dxy, Dxz, Dyz in dipy order
    stack = np.stack([t[..., offd] for t in tensors])         # (N,H,W,D,3)
    masks = np.stack([m > md_thr for m in md_list])
    total = stack.sum(0)
    common = masks.mean(0) > 0.5
    out = []
    for i in range(len(tensors)):
        loo = (total - stack[i]) / max(len(tensors) - 1, 1)
        row = []
        for k in range(3):
            a, b = stack[i][..., k][common], loo[..., k][common]
            row.append(float(np.corrcoef(a, b)[0, 1]) if a.std() > 0 and b.std() > 0 else float("nan"))
        out.append(row)
    return np.array(out)

if USE_AFFINE_HARMONIZATION:
    CACHE_DIR = "/kaggle/working/_volume_cache_affine"
    os.makedirs(CACHE_DIR, exist_ok=True)

    def load_subject_volumes(entry):
        paths = _cache_paths(entry)
        if all(os.path.exists(p) for p in paths.values()):
            try:
                return _load_cached(paths)
            except Exception:
                pass
        try:
            t1, fa, md_norm, tensor_norm = harmonize_affine_arrays(entry)
            np.save(paths["t1"], t1.astype(np.float16))
            np.save(paths["fa"], fa.astype(np.float16))
            np.save(paths["md"], md_norm.astype(np.float16))
            if "tensor" in paths:
                np.save(paths["tensor"], tensor_norm.astype(np.float16))
            return _load_cached(paths)
        except Exception as e:
            print(f"  [WARN] failed to load {entry['group']}:{entry['subject_id']} -> {e}")
            return None
    print(f"Harmonisation: AFFINE-AWARE (world-space, RAS)  cache -> {CACHE_DIR}")
else:
    print("Harmonisation: original Step 3 (voxel-size zoom + centre crop)")
# =====================================================================================

print("IMPORTANT: resampling now uses physical voxel spacing, not just array shape.")
print("If you have an existing _volume_cache/ from a previous run, DELETE it before")
print("re-running this cell — old cached .npy files used the old (incorrect) resampling")
print("and will NOT be automatically regenerated, since the cache-hit check only looks")
print("for file existence, not correctness.")
print()
print("Validating and caching volumes to disk (this reads every file once)...")
train_manifest = filter_loadable(train_manifest, "train")
val_manifest = filter_loadable(val_manifest, "val")
test_manifest = filter_loadable(test_manifest, "test")
print(f"Final counts -> train={len(train_manifest)} val={len(val_manifest)} test={len(test_manifest)}")

if len(train_manifest) == 0 or len(val_manifest) == 0:
    raise RuntimeError("Train or val split is empty after validation — check the [WARN] lines above.")
if "split_df" in globals() and len(test_manifest) != int((split_df.split == "test").sum()):
    raise RuntimeError("A frozen test subject failed to load — fix it rather than silently shrinking the test set.")

## Step 4 - Dataset (3D volumes / patches)

Every volume is already resampled to a fixed canonical shape in Step 3. Training samples
random 3D patches (memory-friendly); validation and test use the full canonical volume
(batch size 1) so evaluation metrics reflect the whole brain, not just a patch.


In [ ]:
# ===================== PATCH P3 — replaces Step 4 ======================================
# Same dataset as before, with three changes (training split only):
#  1. FIXED left-right flip. The old code flipped the volumes but not the tensor, so half the
#     training patches had mirrored anatomy with un-mirrored fibre directions. A reflection of
#     array axis 0 must negate the two tensor components that involve x: Dxy and Dxz.
#  2. T1 intensity augmentation (gamma, smooth bias field, contrast, noise). Tensor untouched.
#  3. AD oversampling: AD subjects are drawn AD_FACTOR times per epoch (default 1.3, i.e. ~100 real
#     AD subjects act like ~130). Extra draws get fresh random augmentation; no new files exist,
#     so nothing can leak into val/test. Validation and test are never augmented.
PATCH_SIZE = (64, 64, 64)  # (H, W, D) training patch, cropped from CANONICAL_SHAPE

OUT_CHANNELS = 8
OUT_NAMES = ['FA', 'MD', 'Dxx', 'Dxy', 'Dyy', 'Dxz', 'Dyz', 'Dzz']   # keep your notebook's list if it differs

AD_FACTOR = 1.3
INTENSITY_AUG = True
FLIP_NEGATE = [OUT_NAMES.index(n) for n in ("Dxy", "Dxz") if n in OUT_NAMES]   # channels whose sign flips


def _random_patch_coords(vol_shape, patch_size):
    coords = []
    for v, p in zip(vol_shape, patch_size):
        max_start = max(0, v - p)
        coords.append(random.randint(0, max_start) if max_start > 0 else 0)
    return coords


def _bias_field(shape, rng, strength=0.3):
    x, y, z = np.meshgrid(*[np.linspace(-1, 1, s, dtype=np.float32) for s in shape], indexing="ij")
    basis = (x, y, z, x * y, x * z, y * z, x * x, y * y, z * z)
    return np.exp(sum(rng.normal(0, strength / 3) * b for b in basis)).astype(np.float32)


def t1_intensity_aug(x, p=0.8):
    """x: (1,H,W,D) T1 in [0,1], background 0. Only brain voxels change."""
    rng = np.random.default_rng(random.getrandbits(32))   # python RNG is re-seeded per DataLoader worker
    brain = x > 0
    if not brain.any():
        return x
    v = x
    if rng.random() < p:
        v = np.where(brain, np.clip(v, 0, None) ** np.exp(rng.uniform(np.log(0.7), np.log(1.5))), 0)
    if rng.random() < p:
        v = v * _bias_field(v.shape[1:], rng)[None]
    if rng.random() < p:
        v = v * rng.uniform(0.9, 1.1) + rng.uniform(-0.05, 0.05)
    if rng.random() < 0.5:
        v = v + rng.normal(0, rng.uniform(0, 0.03), v.shape).astype(np.float32)
    return np.where(brain, np.clip(v, 0, 1), 0).astype(np.float32)


def lr_flip(x, y):
    """Mirror array axis 0 of T1 and targets AND reorient the tensor (negate Dxy, Dxz)."""
    x = x[:, ::-1, :, :].copy()
    y = y[:, ::-1, :, :].copy()
    if FLIP_NEGATE:
        y[FLIP_NEGATE] *= -1
    return x, y


class VolumeDataset(Dataset):
    def __init__(self, manifest_entries, train_mode=True, augment=False):
        self.entries = [e for e in manifest_entries if load_subject_volumes(e) is not None]
        self.train_mode = train_mode
        self.augment = augment
        print(f"  VolumeDataset({'train' if train_mode else 'eval'}): {len(self.entries)} subjects")

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        try:
            entry = self.entries[idx]
            vols = load_subject_volumes(entry)
            if vols is None:
                return None
            t1, fa, md_ = vols["t1"], vols["fa"], vols["md"]
            target_stack = [fa, md_]
            if OUT_CHANNELS > 2:
                tensor = vols["tensor"]
                target_stack += [tensor[..., c] for c in range(6)]
            y_full = np.stack(target_stack, axis=0)
            x_full = t1[None, ...]
            if self.train_mode:
                cx, cy, cz = _random_patch_coords(t1.shape, PATCH_SIZE)
                px, py, pz = PATCH_SIZE
                x = x_full[:, cx:cx+px, cy:cy+py, cz:cz+pz]
                y = y_full[:, cx:cx+px, cy:cy+py, cz:cz+pz]
            else:
                x, y = x_full, y_full
            x = x.astype(np.float32).copy()
            y = y.astype(np.float32).copy()
            if self.train_mode and self.augment:
                if random.random() < 0.5:
                    x, y = lr_flip(x, y)
                if INTENSITY_AUG:
                    x = t1_intensity_aug(x)
            return torch.from_numpy(x), torch.from_numpy(y)
        except Exception as e:
            print(f"  [WARN] sample {idx} failed: {e}")
            return None


class ADOversampler(torch.utils.data.Sampler):
    """Every training subject once per epoch + round((AD_FACTOR-1) * n_AD) extra AD draws, shuffled.
    Advances its own epoch counter, so the training loop needs no changes."""
    def __init__(self, entries, factor=AD_FACTOR, seed=SEED):
        self.n = len(entries)
        self.ad_idx = [i for i, e in enumerate(entries) if e["group"] == "AD"]
        self.n_extra = int(round((factor - 1) * len(self.ad_idx)))
        self.seed, self.epoch = seed, 0

    def __len__(self):
        return self.n + self.n_extra

    def __iter__(self):
        self.epoch += 1
        rng = np.random.default_rng([self.seed, self.epoch])
        extra = rng.choice(self.ad_idx, self.n_extra, replace=self.n_extra > len(self.ad_idx)) if self.n_extra else []
        idx = np.concatenate([np.arange(self.n), np.asarray(extra, dtype=int)])
        rng.shuffle(idx)
        return iter(idx.tolist())


def safe_collate(batch):
    batch = [b for b in batch if b is not None]
    if len(batch) == 0:
        return None
    return torch.stack([b[0] for b in batch]), torch.stack([b[1] for b in batch])


BATCH_SIZE = 2  # kept small deliberately — 3D volumes are GPU-memory hungry
train_ds = VolumeDataset(train_manifest, train_mode=True, augment=True)
val_ds = VolumeDataset(val_manifest, train_mode=False)
test_ds = VolumeDataset(test_manifest, train_mode=False)

train_sampler = ADOversampler(train_ds.entries, AD_FACTOR)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=train_sampler, num_workers=2,
                          pin_memory=True, collate_fn=safe_collate, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=1,
                        pin_memory=True, collate_fn=safe_collate)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=1,
                         pin_memory=True, collate_fn=safe_collate)
print(f"train draws/epoch: {len(train_sampler)} ({train_sampler.n_extra} extra AD draws, factor {AD_FACTOR}); "
      f"flip negates channels {[OUT_NAMES[i] for i in FLIP_NEGATE]}")

## Step 4.5 - Log-Euclidean SPD tensor math

Shared utilities used by the model (Step 5), loss (Step 6), and evaluation (Step 9). The core
idea: a diffusion tensor must be symmetric positive-definite (SPD) to be physically valid, but
a neural network's raw output has no such constraint. The log-Euclidean trick sidesteps this
entirely — the network predicts an *unconstrained* symmetric matrix (the "log-tensor"), and
`symmetric_expm` maps that to a matrix that is **always SPD**, no matter what the network
outputs. No clipping, no rejection sampling, no post-hoc projection back onto the SPD manifold —
the architecture just can't produce an invalid tensor.


In [ ]:
def lower_tri_to_tensor_torch(components):
    """components: (..., 6) in dipy order Dxx, Dxy, Dyy, Dxz, Dyz, Dzz.
    Returns the full symmetric (..., 3, 3) matrix."""
    Dxx, Dxy, Dyy, Dxz, Dyz, Dzz = components.unbind(-1)
    row0 = torch.stack([Dxx, Dxy, Dxz], dim=-1)
    row1 = torch.stack([Dxy, Dyy, Dyz], dim=-1)
    row2 = torch.stack([Dxz, Dyz, Dzz], dim=-1)
    return torch.stack([row0, row1, row2], dim=-2)


def tensor_to_lower_tri_torch(tensor):
    """Inverse of lower_tri_to_tensor_torch. tensor: (..., 3, 3) -> (..., 6)."""
    Dxx = tensor[..., 0, 0]
    Dxy = tensor[..., 0, 1]
    Dxz = tensor[..., 0, 2]
    Dyy = tensor[..., 1, 1]
    Dyz = tensor[..., 1, 2]
    Dzz = tensor[..., 2, 2]
    return torch.stack([Dxx, Dxy, Dyy, Dxz, Dyz, Dzz], dim=-1)


def _regularize_symmetric(tensor, reg=1e-6):
    """Symmetrizes and adds a tiny identity multiple before any eigh call. Guards
    against exactly-singular background/padding voxels."""
    tensor = 0.5 * (tensor + tensor.transpose(-1, -2))
    eye = torch.eye(3, device=tensor.device, dtype=tensor.dtype)
    return tensor + reg * eye


EIGH_CHUNK_SIZE = 32768  # bounds memory per chunk; doesn't change correctness


def _chunked_eigh(matrices, chunk_size=EIGH_CHUNK_SIZE):
    """Runs torch.linalg.eigh in bounded-size chunks, on CPU. PyTorch's batched CUDA
    eigh (cusolverDnXsyevBatched) is broken/unavailable in some Kaggle GPU sessions —
    it fails with CUSOLVER_STATUS_INVALID_VALUE regardless of batch size, even down to
    a single small chunk, which rules out a batch-size limit and points at the cuSOLVER
    backend itself. Routing through CPU (LAPACK) sidesteps that entirely.

    This is safe everywhere it's used in this notebook: during training, eigh is only
    ever applied to the ground-truth tensor (no gradient needed — it's a fixed target,
    not something we backprop through), and every eval/viz call site is already wrapped
    in torch.no_grad(). So a CPU round-trip costs some speed, not correctness or
    gradient flow. Verified numerically identical to a direct GPU call."""
    leading_shape = matrices.shape[:-2]
    flat = matrices.reshape(-1, 3, 3)
    n = flat.shape[0]
    device = flat.device
    eigvals_parts, eigvecs_parts = [], []
    for start in range(0, n, chunk_size):
        end = min(start + chunk_size, n)
        chunk_cpu = flat[start:end].detach().cpu()
        ev, evec = torch.linalg.eigh(chunk_cpu)
        eigvals_parts.append(ev.to(device))
        eigvecs_parts.append(evec.to(device))
    eigvals = torch.cat(eigvals_parts, dim=0).reshape(*leading_shape, 3)
    eigvecs = torch.cat(eigvecs_parts, dim=0).reshape(*leading_shape, 3, 3)
    return eigvals, eigvecs


def spd_safe_logm(tensor, eig_floor=1e-4):
    """Symmetric matrix logarithm with an eigenvalue floor, so near-singular or
    slightly non-positive-definite ground-truth voxels (background, CSF, preprocessing
    noise) never produce -inf/NaN. tensor: (..., 3, 3) symmetric."""
    tensor = _regularize_symmetric(tensor)
    eigvals, eigvecs = _chunked_eigh(tensor)
    eigvals = eigvals.clamp(min=eig_floor)
    log_eigvals = torch.log(eigvals)
    return eigvecs @ torch.diag_embed(log_eigvals) @ eigvecs.transpose(-1, -2)


def symmetric_expm(tensor, eig_cap=20.0):
    """Symmetric matrix exponential — always returns an SPD matrix regardless of the
    input. This is what makes the architecture 'SPD-valid by construction': the network
    can output any real symmetric matrix and this always maps it to a physically valid
    diffusion tensor. eig_cap avoids float overflow for large raw outputs."""
    tensor = _regularize_symmetric(tensor)
    eigvals, eigvecs = _chunked_eigh(tensor)
    exp_eigvals = torch.exp(eigvals.clamp(max=eig_cap))
    return eigvecs @ torch.diag_embed(exp_eigvals) @ eigvecs.transpose(-1, -2)


def tensor_fa_md(tensor, eps=1e-8):
    """tensor: (..., 3, 3) SPD. Returns FA, MD, each (...), derived analytically from
    the tensor's eigenvalues (not predicted as separate channels)."""
    eigvals, _ = _chunked_eigh(_regularize_symmetric(tensor))
    eigvals = eigvals.clamp(min=0)
    md = eigvals.mean(dim=-1)
    diff = eigvals - md.unsqueeze(-1)
    num = (diff ** 2).sum(dim=-1)
    den = (eigvals ** 2).sum(dim=-1) + eps
    fa = torch.sqrt((1.5 * num / den).clamp(min=0))
    return fa, md


def principal_eigvec(tensor):
    """tensor: (..., 3, 3) SPD. Returns (..., 3) unit eigenvector of the largest
    eigenvalue — the principal diffusion direction, used for color-FA and cosine
    similarity."""
    _, eigvecs = _chunked_eigh(_regularize_symmetric(tensor))
    return eigvecs[..., :, -1]

## Step 5 - Model (3D U-Net, log-Euclidean output)

The network's final layer outputs **6 raw, unconstrained values per voxel** — no sigmoid, no
clipping to a physical range. These represent a symmetric matrix in log-tensor space. The
tensor itself only exists after Step 6/9 pass this through `symmetric_expm` — at that point it
is guaranteed SPD. `torch.tanh(...) * LOG_SCALE` just keeps the raw output in a numerically
stable range so `expm` doesn't overflow; it is not the physical constraint (that's `expm`'s job).


In [ ]:
MODEL_OUT_CHANNELS = 6  # raw log-tensor components (Dxx,Dxy,Dyy,Dxz,Dyz,Dzz in log-Euclidean space)
LOG_SCALE = 5.0          # bounds raw output to a numerically stable range before expm


class DoubleConv3D(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1), nn.BatchNorm3d(out_ch), nn.ReLU(inplace=True),
            nn.Conv3d(out_ch, out_ch, 3, padding=1), nn.BatchNorm3d(out_ch), nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class UNet3D(nn.Module):
    """3 levels deep (not 4) and a smaller base width than a typical 2D model — 3D feature
    maps are far more GPU-memory hungry, so this keeps a 64^3 patch training run feasible on
    a single T4/P100. Output is 6 raw log-tensor components, not FA/MD/tensor directly —
    see Step 4.5 and Step 6 for how those get derived."""

    def __init__(self, in_ch=1, out_ch=MODEL_OUT_CHANNELS, base=16):
        super().__init__()
        self.enc1 = DoubleConv3D(in_ch, base)
        self.enc2 = DoubleConv3D(base, base * 2)
        self.enc3 = DoubleConv3D(base * 2, base * 4)
        self.pool = nn.MaxPool3d(2)

        self.bottleneck = DoubleConv3D(base * 4, base * 8)

        self.up3 = nn.ConvTranspose3d(base * 8, base * 4, 2, stride=2)
        self.dec3 = DoubleConv3D(base * 8, base * 4)
        self.up2 = nn.ConvTranspose3d(base * 4, base * 2, 2, stride=2)
        self.dec2 = DoubleConv3D(base * 4, base * 2)
        self.up1 = nn.ConvTranspose3d(base * 2, base, 2, stride=2)
        self.dec1 = DoubleConv3D(base * 2, base)

        self.out_conv = nn.Conv3d(base, out_ch, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b = self.bottleneck(self.pool(e3))

        d3 = self.dec3(torch.cat([self.up3(b), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))

        raw = self.out_conv(d1)
        return torch.tanh(raw) * LOG_SCALE  # bounded, unconstrained log-tensor components


model = UNet3D(in_ch=1, out_ch=MODEL_OUT_CHANNELS, base=16).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"UNet3D parameters: {n_params/1e6:.2f}M")
print("Output: 6 log-Euclidean tensor components. Step 6 maps these through a symmetric "
      "matrix exponential, which guarantees every predicted diffusion tensor is SPD "
      "(physically valid) by construction, not by post-hoc correction.")
print("NOTE: trained on random 64^3 patches; validation/test run on the full "
      f"{CANONICAL_SHAPE} canonical volume (batch size 1).")


## Step 6 - Loss function (log-Euclidean, FA-weighted)

**Log-Euclidean reconstruction loss**: MSE between the model's raw log-tensor output and
`logm(T_gt)` — the ground-truth tensor's matrix logarithm. This is the standard distance metric
for SPD matrices (a plain Euclidean MSE on tensor components is not, since the space of SPD
matrices isn't flat).

**FA-weighted**: each voxel's squared error is weighted by `(FA_gt + eps)`, so white-matter
voxels (high FA, where the diffusion signal is most clinically meaningful) contribute more to
the gradient than background/gray-matter voxels — matching the FA-weighted error term in the
proposed architecture.


In [ ]:
FA_WEIGHT_EPS = 0.1  # floor added to the FA-based per-voxel weight, so background/CSF still gets some gradient


def combined_loss(pred, target):
    """pred: (B, 6, H, W, D) raw log-tensor components from the model.
    target: (B, 8, H, W, D) — channel 0 = FA_gt, channel 1 = MD_gt (unused here, kept for
    dataset/schema consistency with earlier steps), channels 2:8 = ground-truth tensor
    components (scaled, see Step 3).

    The eigendecomposition-based SPD math is numerically sensitive, so it's forced to run
    in float32 regardless of the outer AMP/autocast context — mixed precision here risks
    NaNs from near-degenerate eigenvalues."""
    with torch.autocast(device_type=pred.device.type, enabled=False):
        pred_f = pred.float()
        target_f = target.float()

        fa_gt = target_f[:, 0]                                            # (B,H,W,D)
        tensor_gt_components = target_f[:, 2:8].permute(0, 2, 3, 4, 1)     # (B,H,W,D,6)
        tensor_gt = lower_tri_to_tensor_torch(tensor_gt_components)        # (B,H,W,D,3,3)
        S_gt = spd_safe_logm(tensor_gt)                                    # (B,H,W,D,3,3)
        S_gt_components = tensor_to_lower_tri_torch(S_gt)                  # (B,H,W,D,6)

        S_pred_components = pred_f.permute(0, 2, 3, 4, 1)                  # (B,H,W,D,6)

        weight = (fa_gt + FA_WEIGHT_EPS).unsqueeze(-1)                     # (B,H,W,D,1), broadcasts over 6 channels
        sq_err = (S_pred_components - S_gt_components) ** 2
        loss = (sq_err * weight).mean()

        mse = sq_err.mean().item()  # unweighted, for comparable logging across epochs
    return loss, mse, None


## Step 7 - Training loop

Handles the practical failure modes of a long Kaggle run:
- **Resume-on-restart**: if `checkpoints/last.pt` already exists, training picks up from
  that epoch instead of starting over — just re-run the notebook after a disconnect.
- **NaN/Inf loss**: the batch is skipped (optimizer step not taken), logged, training continues.
- **CUDA OOM on a batch**: cache is cleared and the batch is skipped rather than killing the run.
- **Gradient clipping** to reduce the chance of the above happening at all.
- **Early stopping** on validation loss with configurable patience.
- Per-epoch metrics are appended to `history.csv` (resumable — old rows are kept).


In [ ]:
import os
EXPECT_FRESH_RUN = True

In [ ]:
EPOCHS = 100
LR = 1e-4
PATIENCE = 12
GRAD_CLIP = 1.0

CKPT_DIR = "/kaggle/working/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)
LAST_CKPT = os.path.join(CKPT_DIR, "last.pt")
BEST_CKPT = os.path.join(CKPT_DIR, "best.pt")
HISTORY_CSV = "/kaggle/working/history.csv"

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))

start_epoch = 0
best_val_loss = float("inf")
epochs_no_improve = 0
history = []

if os.path.exists(LAST_CKPT):
    print(f"Found existing checkpoint at {LAST_CKPT} — resuming.")
    ckpt = torch.load(LAST_CKPT, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    scheduler.load_state_dict(ckpt["scheduler_state"])
    start_epoch = ckpt["epoch"] + 1
    best_val_loss = ckpt["best_val_loss"]
    epochs_no_improve = ckpt.get("epochs_no_improve", 0)
    if os.path.exists(HISTORY_CSV):
        import csv
        with open(HISTORY_CSV) as f:
            raw_rows = list(csv.DictReader(f))
        # csv.DictReader returns every value as a string — cast back to numeric types,
        # otherwise mixing these with the int/float values appended later in the same
        # run gives pandas an object-dtype column that matplotlib can't plot.
        history = [
            dict(epoch=int(row["epoch"]), train_loss=float(row["train_loss"]),
                 val_loss=float(row["val_loss"]), val_mse=float(row["val_mse"]),
                 seconds=float(row["seconds"]))
            for row in raw_rows
        ]
    print(f"Resuming from epoch {start_epoch} (best_val_loss={best_val_loss:.5f})")
else:
    print("No checkpoint found — starting fresh.")


def run_epoch(loader, train_mode):
    model.train(train_mode)
    total_loss, total_mse, n_batches, n_skipped = 0.0, 0.0, 0, 0

    for batch in loader:
        if batch is None:
            n_skipped += 1
            continue
        x, y = batch
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)

        try:
            if train_mode:
                optimizer.zero_grad(set_to_none=True)
                with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
                    pred = model(x)
                    loss, mse_val, _ = combined_loss(pred, y)

                if not torch.isfinite(loss):
                    print("  [WARN] non-finite loss, skipping batch")
                    n_skipped += 1
                    continue

                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler.step(optimizer)
                scaler.update()
            else:
                with torch.no_grad(), torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
                    pred = model(x)
                    loss, mse_val, _ = combined_loss(pred, y)
                if not torch.isfinite(loss):
                    n_skipped += 1
                    continue

            total_loss += loss.item()
            total_mse += mse_val
            n_batches += 1

        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                print("  [WARN] CUDA OOM on a batch — clearing cache and skipping it")
                if DEVICE.type == "cuda":
                    torch.cuda.empty_cache()
                n_skipped += 1
                continue
            raise

    if n_skipped:
        print(f"  ({n_skipped} batch(es) skipped this epoch)")
    if n_batches == 0:
        return float("nan"), float("nan")
    return total_loss / n_batches, total_mse / n_batches


print(f"Training from epoch {start_epoch} to {EPOCHS - 1} ...")
for epoch in range(start_epoch, EPOCHS):
    t0 = time.time()
    train_loss, train_mse = run_epoch(train_loader, train_mode=True)
    val_loss, val_mse = run_epoch(val_loader, train_mode=False)
    scheduler.step(val_loss)
    dt = time.time() - t0

    print(f"Epoch {epoch+1}/{EPOCHS} | train_loss={train_loss:.5f} val_loss={val_loss:.5f} "
          f"val_mse={val_mse:.5f} | {dt:.1f}s")

    history.append(dict(epoch=epoch, train_loss=train_loss, val_loss=val_loss,
                         val_mse=val_mse, seconds=dt))
    import csv
    with open(HISTORY_CSV, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["epoch", "train_loss", "val_loss", "val_mse", "seconds"])
        writer.writeheader()
        writer.writerows(history)

    improved = val_loss < best_val_loss
    if improved:
        best_val_loss = val_loss
        epochs_no_improve = 0
        torch.save({"model_state": model.state_dict(), "epoch": epoch, "val_loss": val_loss}, BEST_CKPT)
    else:
        epochs_no_improve += 1

    torch.save({
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "epoch": epoch,
        "best_val_loss": best_val_loss,
        "epochs_no_improve": epochs_no_improve,
    }, LAST_CKPT)

    if epochs_no_improve >= PATIENCE:
        print(f"Early stopping — no improvement for {PATIENCE} epochs.")
        break

print(f"\nTraining done. Best val_loss={best_val_loss:.5f} (checkpoint: {BEST_CKPT})")
print("Reminder: click 'Save Version' periodically on long runs so /kaggle/working/ survives a disconnect.")


## Step 8 - Training curves

In [ ]:
import pandas as pd
hist_df = pd.DataFrame(history)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(hist_df["epoch"], hist_df["train_loss"], label="train")
ax[0].plot(hist_df["epoch"], hist_df["val_loss"], label="val")
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("loss"); ax[0].legend(); ax[0].set_title("Loss")

ax[1].plot(hist_df["epoch"], hist_df["val_mse"], label="val_mse", color="tab:red")
ax[1].set_xlabel("epoch"); ax[1].set_ylabel("MSE"); ax[1].legend(); ax[1].set_title("Validation MSE")
plt.tight_layout()
plt.savefig("/kaggle/working/training_curves.png", dpi=150)
plt.show()


## Step 9 - Test-set evaluation

Loads the best checkpoint and reports, **per group (AD / MCI / HCP) and pooled overall**:

- **FA / MD MSE and SSIM** — derived from the predicted tensor's eigenvalues, compared against
  ground truth
- **Cosine similarity of the principal diffusion direction** — how well the model recovers fiber
  orientation, not just tensor magnitude/shape. Reported both overall and restricted to
  white-matter voxels (`FA_gt > 0.2`), since direction is only clinically meaningful where
  there's real anisotropy to have a direction in the first place

This covers 4 of the 5 metrics from the Review I evaluation slide. Tractography overlap (Dice)
is not included — see the notebook intro for why.


In [ ]:
model.load_state_dict(torch.load(BEST_CKPT, map_location=DEVICE)["model_state"])
model.eval()


def _eval_subject(entry):
    vols = load_subject_volumes(entry)
    if vols is None:
        return None

    t1 = np.asarray(vols["t1"], dtype=np.float32)
    fa_gt_np = np.asarray(vols["fa"], dtype=np.float32)
    md_gt_np = np.asarray(vols["md"], dtype=np.float32)
    tensor_gt_np = np.asarray(vols["tensor"], dtype=np.float32)  # (H,W,D,6)

    x = torch.from_numpy(t1[None, None, ...]).to(DEVICE)
    with torch.no_grad():
        pred_log = model(x)  # (1,6,H,W,D)
        pred_components = pred_log.permute(0, 2, 3, 4, 1)  # (1,H,W,D,6)
        tensor_pred = symmetric_expm(lower_tri_to_tensor_torch(pred_components))  # (1,H,W,D,3,3)
        fa_pred_t, md_pred_t = tensor_fa_md(tensor_pred)
        principal_pred = principal_eigvec(tensor_pred)

        tensor_gt_t = torch.from_numpy(tensor_gt_np).to(DEVICE)[None]  # (1,H,W,D,6)
        tensor_gt_full = lower_tri_to_tensor_torch(tensor_gt_t)
        principal_gt = principal_eigvec(tensor_gt_full)

    fa_pred = fa_pred_t[0].cpu().numpy()
    md_pred = md_pred_t[0].cpu().numpy()

    # abs() because an eigenvector's sign is arbitrary — direction agreement, not sign agreement
    cos_sim = torch.abs((principal_pred[0] * principal_gt[0]).sum(dim=-1)).clamp(max=1.0)
    cos_sim = cos_sim.cpu().numpy()  # (H,W,D)

    fa_mse = float(np.mean((fa_pred - fa_gt_np) ** 2))
    md_mse = float(np.mean((md_pred - md_gt_np) ** 2))

    slice_ssims_fa, slice_ssims_md = [], []
    for si in range(fa_pred.shape[-1]):
        try:
            slice_ssims_fa.append(sk_ssim(fa_pred[..., si], fa_gt_np[..., si], data_range=1.0))
            slice_ssims_md.append(sk_ssim(md_pred[..., si], md_gt_np[..., si], data_range=1.0))
        except Exception:
            continue

    wm_mask = fa_gt_np > 0.2
    cos_overall = float(cos_sim.mean())
    cos_wm = float(cos_sim[wm_mask].mean()) if wm_mask.any() else None

    return dict(
        fa_mse=fa_mse, md_mse=md_mse,
        fa_ssim=float(np.mean(slice_ssims_fa)) if slice_ssims_fa else None,
        md_ssim=float(np.mean(slice_ssims_md)) if slice_ssims_md else None,
        cosine_overall=cos_overall, cosine_white_matter=cos_wm,
    )


per_subject = []
for entry in test_ds.entries:
    r = _eval_subject(entry)
    if r is None:
        print(f"  [WARN] eval failed for {entry['group']}:{entry['subject_id']}")
        continue
    r["group"] = entry["group"]
    r["subject_id"] = entry["subject_id"]
    per_subject.append(r)


def _aggregate(rows):
    if not rows:
        return None
    out = {}
    for key in ["fa_mse", "md_mse", "fa_ssim", "md_ssim", "cosine_overall", "cosine_white_matter"]:
        vals = [r[key] for r in rows if r.get(key) is not None]
        out[key + "_mean"] = float(np.mean(vals)) if vals else None
        out[key + "_std"] = float(np.std(vals)) if vals else None
    out["n_subjects"] = len(rows)
    return out


results = {"overall": _aggregate(per_subject)}
for g in ("AD", "MCI", "CN", "HCP"):
    rows = [r for r in per_subject if r["group"] == g]
    results[g] = _aggregate(rows)

print(json.dumps(results, indent=2))
with open("/kaggle/working/test_results.json", "w") as f:
    json.dump(results, f, indent=2)
with open("/kaggle/working/test_results_per_subject.json", "w") as f:
    json.dump(per_subject, f, indent=2)


In [ ]:
import os

model.eval()


def _resolve_mask_ext(subj_dir, base_name="brain_mask"):
    for ext in (".nii.gz", ".nii"):
        candidate = os.path.join(subj_dir, base_name + ext)
        if os.path.exists(candidate):
            return candidate
    return None


def load_brain_mask(entry, t1_np):
    """Loads and resamples this subject's brain_mask to the canonical grid, using
    the same physical-spacing-aware resampling as every other volume (Step 3) but
    with nearest-neighbor interpolation (order=0), since a mask is binary/categorical,
    not continuous intensity -- linear interpolation would blur its edges into
    fractional values. Falls back to thresholding the already-loaded T1 volume if no
    brain_mask file is found (T1 is already skull-stripped, so background is ~0)."""
    subj_dir = os.path.dirname(entry["t1"])
    mask_path = _resolve_mask_ext(subj_dir)
    if mask_path is not None:
        try:
            mask_raw, mask_zooms = _load_nifti(mask_path)
            mask_raw = _clean(mask_raw)
            mask_resampled = _resample(mask_raw, mask_zooms, CANONICAL_SHAPE, order=0)
            return mask_resampled > 0.5
        except Exception as e:
            print(f"  [WARN] failed to load brain_mask for {entry['group']}:{entry['subject_id']} -> {e}; falling back to T1 threshold")
    else:
        print(f"  [INFO] no brain_mask file found for {entry['group']}:{entry['subject_id']} -- using T1-threshold fallback")
    return t1_np > 0.01


def _eval_subject_masked(entry):
    vols = load_subject_volumes(entry)
    if vols is None:
        return None

    t1 = np.asarray(vols["t1"], dtype=np.float32)
    fa_gt_np = np.asarray(vols["fa"], dtype=np.float32)
    md_gt_np = np.asarray(vols["md"], dtype=np.float32)
    tensor_gt_np = np.asarray(vols["tensor"], dtype=np.float32)

    brain_mask = load_brain_mask(entry, t1)  # (H,W,D) bool

    x = torch.from_numpy(t1[None, None, ...]).to(DEVICE)
    with torch.no_grad():
        pred_log = model(x)
        pred_components = pred_log.permute(0, 2, 3, 4, 1)
        tensor_pred = symmetric_expm(lower_tri_to_tensor_torch(pred_components))
        fa_pred_t, md_pred_t = tensor_fa_md(tensor_pred)
        principal_pred = principal_eigvec(tensor_pred)

        tensor_gt_t = torch.from_numpy(tensor_gt_np).to(DEVICE)[None]
        tensor_gt_full = lower_tri_to_tensor_torch(tensor_gt_t)
        principal_gt = principal_eigvec(tensor_gt_full)

    fa_pred = fa_pred_t[0].cpu().numpy()
    md_pred = md_pred_t[0].cpu().numpy()

    cos_sim = torch.abs((principal_pred[0] * principal_gt[0]).sum(dim=-1)).clamp(max=1.0)
    cos_sim = cos_sim.cpu().numpy()

    # MSE and cosine similarity: true voxel-level masking, since these are simple
    # elementwise reductions (no spatial windowing involved).
    if brain_mask.any():
        fa_mse = float(np.mean((fa_pred[brain_mask] - fa_gt_np[brain_mask]) ** 2))
        md_mse = float(np.mean((md_pred[brain_mask] - md_gt_np[brain_mask]) ** 2))
    else:
        fa_mse = md_mse = None

    # SSIM: skimage's windowed SSIM needs a rectangular image, not an arbitrary voxel
    # subset, so true per-voxel masking isn't directly possible. Instead, zero out
    # BOTH prediction and ground truth outside the brain mask before computing the
    # standard windowed SSIM per slice -- ground truth is already ~0 there (skull-
    # stripped), so this removes the log-Euclidean architecture's background-bleed
    # penalty (Section X) without needing masked-window statistics.
    fa_pred_masked = np.where(brain_mask, fa_pred, 0.0)
    fa_gt_masked = np.where(brain_mask, fa_gt_np, 0.0)
    md_pred_masked = np.where(brain_mask, md_pred, 0.0)
    md_gt_masked = np.where(brain_mask, md_gt_np, 0.0)

    slice_ssims_fa, slice_ssims_md = [], []
    for si in range(fa_pred.shape[-1]):
        if not brain_mask[..., si].any():
            continue  # skip slices with no brain tissue at all
        try:
            slice_ssims_fa.append(sk_ssim(fa_pred_masked[..., si], fa_gt_masked[..., si], data_range=1.0))
            slice_ssims_md.append(sk_ssim(md_pred_masked[..., si], md_gt_masked[..., si], data_range=1.0))
        except Exception:
            continue

    wm_mask = (fa_gt_np > 0.2) & brain_mask
    cos_overall = float(cos_sim[brain_mask].mean()) if brain_mask.any() else None
    cos_wm = float(cos_sim[wm_mask].mean()) if wm_mask.any() else None

    return dict(
        fa_mse=fa_mse, md_mse=md_mse,
        fa_ssim=float(np.mean(slice_ssims_fa)) if slice_ssims_fa else None,
        md_ssim=float(np.mean(slice_ssims_md)) if slice_ssims_md else None,
        cosine_overall=cos_overall, cosine_white_matter=cos_wm,
        brain_voxel_frac=float(brain_mask.mean()),
    )


per_subject_masked = []
for entry in test_ds.entries:
    r = _eval_subject_masked(entry)
    if r is None:
        print(f"  [WARN] masked eval failed for {entry['group']}:{entry['subject_id']}")
        continue
    r["group"] = entry["group"]
    r["subject_id"] = entry["subject_id"]
    per_subject_masked.append(r)


def _aggregate(rows):
    if not rows:
        return None
    out = {}
    for key in ["fa_mse", "md_mse", "fa_ssim", "md_ssim", "cosine_overall", "cosine_white_matter"]:
        vals = [r[key] for r in rows if r.get(key) is not None]
        out[key + "_mean"] = float(np.mean(vals)) if vals else None
        out[key + "_std"] = float(np.std(vals)) if vals else None
    out["n_subjects"] = len(rows)
    return out


results_masked = {"overall": _aggregate(per_subject_masked)}
for g in ("AD", "MCI", "CN", "HCP"):
    rows = [r for r in per_subject_masked if r["group"] == g]
    results_masked[g] = _aggregate(rows)

print(json.dumps(results_masked, indent=2))

with open("/kaggle/working/test_results_masked.json", "w") as f:
    json.dump(results_masked, f, indent=2)
with open("/kaggle/working/test_results_masked_per_subject.json", "w") as f:
    json.dump(per_subject_masked, f, indent=2)

# --- Side-by-side comparison against your existing whole-volume results ---
existing_path = "/kaggle/working/test_results.json"
if os.path.exists(existing_path):
    with open(existing_path) as f:
        results_unmasked = json.load(f)
    print("\n" + "=" * 70)
    print(f"{'Metric':<20}{'Whole-volume':>15}{'Brain-masked':>15}{'Change':>15}")
    print("=" * 70)
    for key in ["fa_mse_mean", "md_mse_mean", "fa_ssim_mean", "md_ssim_mean",
                "cosine_overall_mean", "cosine_white_matter_mean"]:
        old_v = results_unmasked["overall"].get(key)
        new_v = results_masked["overall"].get(key)
        if old_v is not None and new_v is not None:
            print(f"{key:<20}{old_v:>15.4f}{new_v:>15.4f}{new_v - old_v:>+15.4f}")

## Step 10 - Qualitative check: T1 vs. ground truth vs. prediction vs. error

Two figures:
1. **T1 / GT / Prediction / Error** for FA and MD — the error panel uses a diverging
   colormap centered at zero, so blue/red = under/over-prediction and white = accurate.
2. **Color-FA** (standard DTI fiber-direction visualization: RGB = principal diffusion
   direction, brightness = FA) — only possible in the tensor-output notebooks, since it's
   reconstructed from the predicted diffusion tensor's principal eigenvector. In the FA+MD-only
   notebooks this section is skipped with an explanation, since there's no predicted tensor to
   derive a direction from.


In [ ]:
def lower_tri_to_tensor(components):
    """components: (..., 6) in dipy's lower_triangular order: Dxx, Dxy, Dyy, Dxz, Dyz, Dzz.
    Returns the full symmetric (..., 3, 3) diffusion tensor."""
    Dxx, Dxy, Dyy, Dxz, Dyz, Dzz = [components[..., i] for i in range(6)]
    row0 = np.stack([Dxx, Dxy, Dxz], axis=-1)
    row1 = np.stack([Dxy, Dyy, Dyz], axis=-1)
    row2 = np.stack([Dxz, Dyz, Dzz], axis=-1)
    return np.stack([row0, row1, row2], axis=-2)


def color_fa_map(fa_slice, tensor_components_slice):
    """fa_slice: (H, W) in [0, 1]. tensor_components_slice: (H, W, 6).
    Returns an (H, W, 3) RGB array: direction from the tensor's principal eigenvector,
    brightness scaled by FA. Eigenvector direction is scale-invariant, so this works
    whether the tensor components are physically scaled or normalized."""
    tensor = lower_tri_to_tensor(tensor_components_slice)
    # np.linalg.eigh returns ascending eigenvalues; the last eigenvector is principal
    _, eigvecs = np.linalg.eigh(tensor)
    principal = eigvecs[..., :, -1]
    color = np.abs(principal) * fa_slice[..., None]
    return np.clip(color, 0, 1)


def plot_t1_gt_pred_error(x2d, y2d, pred2d, out_names, n_show, save_path):
    """x2d: (B,1,H,W), y2d/pred2d: (B,C,H,W). Plots T1 / GT / Pred / Error for the first
    two channels (FA, MD) across n_show rows."""
    n_channels_to_show = min(2, len(out_names))
    n_cols = 1 + 3 * n_channels_to_show
    fig, axes = plt.subplots(n_show, n_cols, figsize=(4 * n_cols, 4 * n_show))
    if n_show == 1:
        axes = axes[None, :]

    for row in range(n_show):
        axes[row, 0].imshow(x2d[row, 0], cmap="gray")
        axes[row, 0].set_title("T1"); axes[row, 0].axis("off")
        col = 1
        for ci in range(n_channels_to_show):
            name = out_names[ci]
            gt, pred = y2d[row, ci], pred2d[row, ci]
            err = pred - gt
            vmax = max(np.abs(err).max(), 1e-6)

            axes[row, col].imshow(gt, cmap="hot")
            axes[row, col].set_title(f"GT {name}"); axes[row, col].axis("off")

            axes[row, col + 1].imshow(pred, cmap="hot")
            axes[row, col + 1].set_title(f"Pred {name}"); axes[row, col + 1].axis("off")

            im = axes[row, col + 2].imshow(err, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
            axes[row, col + 2].set_title(f"Error {name} (pred-GT)"); axes[row, col + 2].axis("off")
            plt.colorbar(im, ax=axes[row, col + 2], fraction=0.046, pad=0.04)

            col += 3

    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.show()


def plot_color_fa(y2d, pred2d, out_names, n_show, save_path):
    """Only valid when tensor channels are present (OUT_CHANNELS > 2).
    Reconstructs color-FA from channels [FA] + [Dxx..Dzz] for both GT and prediction."""
    if len(out_names) <= 2:
        print("Color-FA skipped: this notebook only predicts FA + MD, so there's no "
              "predicted diffusion tensor to derive a fiber direction from. Use one of "
              "the *_fa_md_tensor notebooks for color-FA visualization.")
        return

    fig, axes = plt.subplots(n_show, 2, figsize=(8, 4 * n_show))
    if n_show == 1:
        axes = axes[None, :]

    for row in range(n_show):
        fa_gt = y2d[row, 0]
        fa_pred = pred2d[row, 0]
        tensor_gt = np.stack([y2d[row, 2 + c] for c in range(6)], axis=-1)
        tensor_pred = np.stack([pred2d[row, 2 + c] for c in range(6)], axis=-1)

        cfa_gt = color_fa_map(fa_gt, tensor_gt)
        cfa_pred = color_fa_map(fa_pred, tensor_pred)

        axes[row, 0].imshow(cfa_gt); axes[row, 0].set_title("GT Color-FA"); axes[row, 0].axis("off")
        axes[row, 1].imshow(cfa_pred); axes[row, 1].set_title("Pred Color-FA"); axes[row, 1].axis("off")

    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.show()

def tensor_to_lower_tri_np(tensor):
    """Inverse of lower_tri_to_tensor. tensor: (..., 3, 3) -> (..., 6)."""
    Dxx = tensor[..., 0, 0]
    Dxy = tensor[..., 0, 1]
    Dxz = tensor[..., 0, 2]
    Dyy = tensor[..., 1, 1]
    Dyz = tensor[..., 1, 2]
    Dzz = tensor[..., 2, 2]
    return np.stack([Dxx, Dxy, Dyy, Dxz, Dyz, Dzz], axis=-1)


def symmetric_expm_np(mat):
    """numpy version of the model's SPD-guaranteeing symmetric matrix exponential —
    used here to turn the raw 6-channel model output back into a valid tensor for
    visualization, mirroring what Step 6/9 do in torch."""
    mat = 0.5 * (mat + np.swapaxes(mat, -1, -2))
    eigvals, eigvecs = np.linalg.eigh(mat)
    exp_eigvals = np.exp(np.clip(eigvals, None, 20.0))
    return np.einsum("...ij,...j,...kj->...ik", eigvecs, exp_eigvals, eigvecs)


def tensor_fa_md_np(tensor):
    """tensor: (..., 3, 3) SPD. Returns FA, MD derived from eigenvalues."""
    eigvals = np.clip(np.linalg.eigvalsh(tensor), 0, None)
    md = eigvals.mean(axis=-1)
    diff = eigvals - md[..., None]
    num = (diff ** 2).sum(axis=-1)
    den = (eigvals ** 2).sum(axis=-1) + 1e-8
    fa = np.sqrt(np.clip(1.5 * num / den, 0, None))
    return fa, md


def raw_pred_to_full_channels(pred_log):
    """pred_log: (B,6,H,W,D) raw model output (log-tensor components). Returns
    (B,8,H,W,D): FA, MD + 6 tensor components — the same 8-channel layout as the
    ground-truth target, so the plotting helpers above work unchanged even though the
    model itself only outputs 6 raw channels."""
    components = np.moveaxis(pred_log, 1, -1)             # (B,H,W,D,6)
    tensor = lower_tri_to_tensor(components)               # (B,H,W,D,3,3)
    tensor_spd = symmetric_expm_np(tensor)                 # (B,H,W,D,3,3), always SPD
    fa, md = tensor_fa_md_np(tensor_spd)                   # each (B,H,W,D)
    tensor_components = tensor_to_lower_tri_np(tensor_spd)  # (B,H,W,D,6)
    full = np.concatenate([fa[..., None], md[..., None], tensor_components], axis=-1)  # (B,H,W,D,8)
    return np.moveaxis(full, -1, 1)  # (B,8,H,W,D)


In [ ]:
sample_batch = None
for batch in test_loader:
    if batch is not None:
        sample_batch = batch
        break

if sample_batch is None:
    print("No test samples available to visualize.")
else:
    x, y = sample_batch
    with torch.no_grad():
        pred_log = model(x.to(DEVICE)).cpu().numpy()  # (B,6,H,W,D) raw log-tensor output
    pred_full = raw_pred_to_full_channels(pred_log)     # (B,8,H,W,D): FA,MD + 6 tensor components
    x = x.numpy()
    y = y.numpy()

    mid = x.shape[-1] // 2  # mid-axial slice of the canonical volume
    x2d = x[..., mid]         # (B, 1, H, W)
    y2d = y[..., mid]         # (B, 8, H, W)
    pred2d = pred_full[..., mid]  # (B, 8, H, W)
    n_show = 1  # 3D val/test batch size is 1

    plot_t1_gt_pred_error(x2d, y2d, pred2d, OUT_NAMES, n_show, "/kaggle/working/qualitative_check.png")
    plot_color_fa(y2d, pred2d, OUT_NAMES, n_show, "/kaggle/working/color_fa_check.png")


## Step 11 - Bidirectional Synthesis Check

From the Review I slides: *"If MRI -> DTI -> MRI' closely matches the original MRI, the
synthetic DTI preserved real anatomical information rather than introducing artifacts."*

Trains a second, lightweight reverse U-Net (tensor -> T1) on the same train/val split, reusing
the existing data loaders by simply swapping input and target (`y[:,2:8]` tensor components in,
`x` T1 out — no new Dataset needed). At test time: take the **forward** model's synthetic
tensor for each test subject, run it through the **reverse** model to get MRI′, and compare
MRI′ against that subject's real T1.

**Caveat:** this reverse model gets far less training budget (fewer epochs, no
architecture tuning) than the forward model — it exists to sanity-check anatomical
information preservation, not as a polished result in its own right.


In [ ]:
rev_ckpt_path = "/kaggle/working/checkpoints_reverse/last.pt"
if EXPECT_FRESH_RUN and os.path.exists(rev_ckpt_path):
    raise RuntimeError(f"Expected a fresh run but found an existing reverse-model checkpoint at {rev_ckpt_path}.")

In [ ]:
class ReverseUNet3D(nn.Module):
    """Mirrors UNet3D's structure but maps tensor components -> T1. Deliberately smaller
    (base=12) since this is a supporting sanity-check model, not the main result."""

    def __init__(self, in_ch=6, out_ch=1, base=12):
        super().__init__()
        self.enc1 = DoubleConv3D(in_ch, base)
        self.enc2 = DoubleConv3D(base, base * 2)
        self.enc3 = DoubleConv3D(base * 2, base * 4)
        self.pool = nn.MaxPool3d(2)

        self.bottleneck = DoubleConv3D(base * 4, base * 8)

        self.up3 = nn.ConvTranspose3d(base * 8, base * 4, 2, stride=2)
        self.dec3 = DoubleConv3D(base * 8, base * 4)
        self.up2 = nn.ConvTranspose3d(base * 4, base * 2, 2, stride=2)
        self.dec2 = DoubleConv3D(base * 4, base * 2)
        self.up1 = nn.ConvTranspose3d(base * 2, base, 2, stride=2)
        self.dec1 = DoubleConv3D(base * 2, base)

        self.out_conv = nn.Conv3d(base, out_ch, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b = self.bottleneck(self.pool(e3))
        d3 = self.dec3(torch.cat([self.up3(b), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return torch.sigmoid(self.out_conv(d1))  # T1 was normalized to [0,1] in Step 3


reverse_model = ReverseUNet3D(in_ch=6, out_ch=1, base=12).to(DEVICE)
print(f"ReverseUNet3D parameters: {sum(p.numel() for p in reverse_model.parameters())/1e6:.2f}M")

REV_EPOCHS = 30
REV_LR = 1e-4
REV_PATIENCE = 6
REV_CKPT_DIR = "/kaggle/working/checkpoints_reverse"
os.makedirs(REV_CKPT_DIR, exist_ok=True)
REV_LAST_CKPT = os.path.join(REV_CKPT_DIR, "last.pt")
REV_BEST_CKPT = os.path.join(REV_CKPT_DIR, "best.pt")

rev_optimizer = torch.optim.Adam(reverse_model.parameters(), lr=REV_LR)
rev_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(rev_optimizer, mode="min", factor=0.5, patience=3)
rev_scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))

rev_start_epoch = 0
rev_best_val_loss = float("inf")
rev_epochs_no_improve = 0

if os.path.exists(REV_LAST_CKPT):
    print(f"Found existing reverse-model checkpoint — resuming.")
    ckpt = torch.load(REV_LAST_CKPT, map_location=DEVICE)
    reverse_model.load_state_dict(ckpt["model_state"])
    rev_optimizer.load_state_dict(ckpt["optimizer_state"])
    rev_scheduler.load_state_dict(ckpt["scheduler_state"])
    rev_start_epoch = ckpt["epoch"] + 1
    rev_best_val_loss = ckpt["best_val_loss"]
    rev_epochs_no_improve = ckpt.get("epochs_no_improve", 0)
    print(f"Resuming reverse model from epoch {rev_start_epoch} (best_val_loss={rev_best_val_loss:.5f})")
else:
    print("No reverse-model checkpoint found — starting fresh.")


def run_reverse_epoch(loader, train_mode):
    reverse_model.train(train_mode)
    total_loss, n_batches, n_skipped = 0.0, 0, 0
    for batch in loader:
        if batch is None:
            n_skipped += 1
            continue
        x, y = batch
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        rev_input = y[:, 2:8]   # ground-truth tensor components -> reverse model input
        rev_target = x          # T1 -> reverse model target

        try:
            if train_mode:
                rev_optimizer.zero_grad(set_to_none=True)
                with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
                    pred_t1 = reverse_model(rev_input)
                    loss = F.mse_loss(pred_t1, rev_target)
                if not torch.isfinite(loss):
                    n_skipped += 1
                    continue
                rev_scaler.scale(loss).backward()
                rev_scaler.unscale_(rev_optimizer)
                torch.nn.utils.clip_grad_norm_(reverse_model.parameters(), 1.0)
                rev_scaler.step(rev_optimizer)
                rev_scaler.update()
            else:
                with torch.no_grad(), torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
                    pred_t1 = reverse_model(rev_input)
                    loss = F.mse_loss(pred_t1, rev_target)
                if not torch.isfinite(loss):
                    n_skipped += 1
                    continue
            total_loss += loss.item()
            n_batches += 1
        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                if DEVICE.type == "cuda":
                    torch.cuda.empty_cache()
                n_skipped += 1
                continue
            raise
    if n_batches == 0:
        return float("nan")
    return total_loss / n_batches


print(f"Training reverse model from epoch {rev_start_epoch} to {REV_EPOCHS - 1} ...")
for epoch in range(rev_start_epoch, REV_EPOCHS):
    t0 = time.time()
    rev_train_loss = run_reverse_epoch(train_loader, train_mode=True)
    rev_val_loss = run_reverse_epoch(val_loader, train_mode=False)
    rev_scheduler.step(rev_val_loss)
    dt = time.time() - t0
    print(f"[reverse] Epoch {epoch+1}/{REV_EPOCHS} | train_loss={rev_train_loss:.5f} "
          f"val_loss={rev_val_loss:.5f} | {dt:.1f}s")

    improved = rev_val_loss < rev_best_val_loss
    if improved:
        rev_best_val_loss = rev_val_loss
        rev_epochs_no_improve = 0
        torch.save({"model_state": reverse_model.state_dict(), "epoch": epoch,
                    "val_loss": rev_val_loss}, REV_BEST_CKPT)
    else:
        rev_epochs_no_improve += 1

    torch.save({
        "model_state": reverse_model.state_dict(),
        "optimizer_state": rev_optimizer.state_dict(),
        "scheduler_state": rev_scheduler.state_dict(),
        "epoch": epoch,
        "best_val_loss": rev_best_val_loss,
        "epochs_no_improve": rev_epochs_no_improve,
    }, REV_LAST_CKPT)

    if rev_epochs_no_improve >= REV_PATIENCE:
        print(f"[reverse] Early stopping — no improvement for {REV_PATIENCE} epochs.")
        break

print(f"Reverse model done. Best val_loss={rev_best_val_loss:.5f}")


In [ ]:
reverse_model.load_state_dict(torch.load(REV_BEST_CKPT, map_location=DEVICE)["model_state"])
reverse_model.eval()
model.eval()


def _bidirectional_check(entry):
    vols = load_subject_volumes(entry)
    if vols is None:
        return None
    t1 = np.asarray(vols["t1"], dtype=np.float32)

    x = torch.from_numpy(t1[None, None, ...]).to(DEVICE)
    with torch.no_grad():
        pred_log = model(x)
        pred_components = pred_log.permute(0, 2, 3, 4, 1)
        tensor_pred = symmetric_expm(lower_tri_to_tensor_torch(pred_components))
        tensor_pred_components = tensor_to_lower_tri_torch(tensor_pred).permute(0, 4, 1, 2, 3)
        mri_prime = reverse_model(tensor_pred_components)[0, 0].cpu().numpy()

    mse = float(np.mean((mri_prime - t1) ** 2))
    slice_ssims = []
    for si in range(t1.shape[-1]):
        try:
            slice_ssims.append(sk_ssim(mri_prime[..., si], t1[..., si], data_range=1.0))
        except Exception:
            continue
    ssim_val = float(np.mean(slice_ssims)) if slice_ssims else None
    return dict(mri_prime_mse=mse, mri_prime_ssim=ssim_val)


bidir_per_subject = []
for entry in test_ds.entries:
    r = _bidirectional_check(entry)
    if r is None:
        print(f"  [WARN] bidirectional check failed for {entry['group']}:{entry['subject_id']}")
        continue
    r["group"] = entry["group"]
    r["subject_id"] = entry["subject_id"]
    bidir_per_subject.append(r)


def _agg_bidir(rows):
    if not rows:
        return None
    out = {}
    for key in ["mri_prime_mse", "mri_prime_ssim"]:
        vals = [r[key] for r in rows if r.get(key) is not None]
        out[key + "_mean"] = float(np.mean(vals)) if vals else None
        out[key + "_std"] = float(np.std(vals)) if vals else None
    out["n_subjects"] = len(rows)
    return out


bidirectional_results = {"overall": _agg_bidir(bidir_per_subject)}
for g in ("AD", "MCI", "CN", "HCP"):
    bidirectional_results[g] = _agg_bidir([r for r in bidir_per_subject if r["group"] == g])

print(json.dumps(bidirectional_results, indent=2))
with open("/kaggle/working/bidirectional_check_results.json", "w") as f:
    json.dump(bidirectional_results, f, indent=2)


## Step 12 - Diagnostic Validity Check

From the Review I slides: run the same CN/MCI/AD classifier on real DTI and on synthetic DTI,
compare accuracy.

Now that real ADNI-CN subjects are in the dataset, this uses the **actual CN/MCI/AD groups**,
not the HCP stand-in from earlier versions of this notebook — HCP is excluded from this
specific analysis (it stays in the main U-Net training pool for image synthesis, just not in
this diagnostic classification check, since it isn't part of ADNI's diagnostic taxonomy).

**Caveat:** with only ~30 CN subjects (likely fewer after the train/val/test split), the CN
class is the smallest of the three — expect noisier per-class metrics for CN specifically than
for AD/MCI. Treat accuracy numbers as illustrative/comparative (real-DTI accuracy vs.
synthetic-DTI accuracy), not as a statistically powered clinical result.

Uses simple tabular features (mean/std FA and MD, whole-brain and white-matter-restricted)
rather than a full 3D CNN classifier — with a test set this small, a from-scratch deep
classifier would be too data-starved to give a meaningful signal.


In [ ]:
!pip install -q scikit-learn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

RNG_SEED = 42


def _extract_features(fa_vol, md_vol):
    wm_mask = fa_vol > 0.2
    wm_frac = float(wm_mask.mean())
    feats = [
        float(fa_vol.mean()), float(fa_vol.std()),
        float(md_vol.mean()), float(md_vol.std()),
        float(fa_vol[wm_mask].mean()) if wm_mask.any() else 0.0,
        float(md_vol[wm_mask].mean()) if wm_mask.any() else 0.0,
        wm_frac,
    ]
    return feats


FEATURE_NAMES = ["fa_mean", "fa_std", "md_mean", "md_std", "wm_fa_mean", "wm_md_mean", "wm_frac"]

model.eval()


def _subject_features(entry):
    vols = load_subject_volumes(entry)
    if vols is None:
        return None
    t1 = np.asarray(vols["t1"], dtype=np.float32)
    fa_gt = np.asarray(vols["fa"], dtype=np.float32)
    md_gt = np.asarray(vols["md"], dtype=np.float32)

    gt_feats = _extract_features(fa_gt, md_gt)

    x = torch.from_numpy(t1[None, None, ...]).to(DEVICE)
    with torch.no_grad():
        pred_log = model(x)
        pred_components = pred_log.permute(0, 2, 3, 4, 1)
        tensor_pred = symmetric_expm(lower_tri_to_tensor_torch(pred_components))
        fa_pred_t, md_pred_t = tensor_fa_md(tensor_pred)
    fa_pred = fa_pred_t[0].cpu().numpy()
    md_pred = md_pred_t[0].cpu().numpy()
    synth_feats = _extract_features(fa_pred, md_pred)

    return dict(group=entry["group"], subject_id=entry["subject_id"],
                gt_features=gt_feats, synth_features=synth_feats)


DIAGNOSTIC_GROUPS = ("AD", "MCI", "CN")  # true ADNI diagnostic groups only — HCP excluded here

trainval_entries = [e for e in (train_ds.entries + val_ds.entries) if e["group"] in DIAGNOSTIC_GROUPS]
test_entries = [e for e in test_ds.entries if e["group"] in DIAGNOSTIC_GROUPS]
trainval_data = [f for f in (_subject_features(e) for e in trainval_entries) if f is not None]
test_data = [f for f in (_subject_features(e) for e in test_entries) if f is not None]

print(f"Classifier train+val subjects: {len(trainval_data)} | test subjects: {len(test_data)}")


def _run_classifier(feature_key):
    X_train = np.array([d[feature_key] for d in trainval_data])
    y_train = np.array([d["group"] for d in trainval_data])
    X_test = np.array([d[feature_key] for d in test_data])
    y_test = np.array([d["group"] for d in test_data])

    clf = RandomForestClassifier(n_estimators=200, max_depth=4, random_state=RNG_SEED, class_weight="balanced")
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    acc = float(accuracy_score(y_test, y_pred))
    report = classification_report(y_test, y_pred, zero_division=0, output_dict=True)
    return dict(accuracy=acc, report=report, n_train=len(y_train), n_test=len(y_test))


diagnostic_results = {}
if len(trainval_data) >= 3 and len(test_data) >= 1:
    diagnostic_results["real_dti"] = _run_classifier("gt_features")
    diagnostic_results["synthetic_dti"] = _run_classifier("synth_features")
    print(f"Real-DTI classifier accuracy:      {diagnostic_results['real_dti']['accuracy']:.3f}")
    print(f"Synthetic-DTI classifier accuracy: {diagnostic_results['synthetic_dti']['accuracy']:.3f}")
else:
    print("Not enough subjects to train/evaluate a classifier meaningfully — skipping.")
    diagnostic_results = None

with open("/kaggle/working/diagnostic_validity_results.json", "w") as f:
    json.dump(diagnostic_results, f, indent=2)


## Step 13 - Tractography Overlap (Dice)

The Review I slides call for "Dice score / volume overlap between fiber tracts reconstructed
from real vs. synthetic tensors."

**This is a simplified, custom deterministic tracker, not a full `dipy` probabilistic
pipeline** — `dipy`'s standard tracking tools (`LocalTracking`, `peaks_from_model`) are built
around raw diffusion-weighted signal + a gradient table, which this pipeline doesn't have (we
only ever have fitted tensors, real or synthetic). Instead: seed points are drawn from
high-FA voxels, and streamlines are propagated by stepping along the tensor's principal
eigenvector direction (nearest-neighbor field sampling, not trilinear — a further
simplification), stopping when FA drops below threshold, the streamline exits the volume, or
the direction changes too sharply. The same seed points are tracked through both the
ground-truth and the synthetic tensor field for each test subject, both streamline sets are
rasterized to binary voxel-occupancy masks, and Dice is computed between those two masks.

This gives a genuine coarse volume-overlap signal, but is **not equivalent to a validated
tractography pipeline** — treat it as a directional/relative comparison, not an
absolute clinical-grade metric, and say so explicitly if this number goes in the paper.


In [ ]:
def _sample_nearest(field, pos, shape):
    """field: (H,W,D,...) array. pos: (N,3) float voxel coords. Nearest-neighbor sample —
    a simplification vs. trilinear interpolation, in line with this being a coarse,
    illustrative tracker rather than a validated pipeline."""
    idx = np.round(pos).astype(int)
    valid = np.all((idx >= 0) & (idx < np.array(shape)), axis=-1)
    idx_clamped = np.clip(idx, 0, np.array(shape) - 1)
    vals = field[idx_clamped[:, 0], idx_clamped[:, 1], idx_clamped[:, 2]]
    return vals, valid


def _track_streamlines(principal_field, fa_field, seeds, step_size=0.6, max_steps=120,
                        fa_thresh=0.15, angle_thresh_deg=60):
    """Vectorized across all seeds simultaneously — each seed is one streamline, all
    stepped together for speed. Returns (n_seeds, max_steps+1, 3) positions; a streamline
    that stops early just stays frozen at its last valid position for remaining steps."""
    shape = fa_field.shape
    n = seeds.shape[0]
    pos = seeds.astype(np.float32).copy()
    prev_dir = np.zeros((n, 3), dtype=np.float32)
    active = np.ones(n, dtype=bool)
    all_points = [pos.copy()]

    for step in range(max_steps):
        dirs, valid = _sample_nearest(principal_field, pos, shape)
        fa_vals, _ = _sample_nearest(fa_field, pos, shape)

        dot = np.sum(dirs * prev_dir, axis=-1)
        flip = dot < 0
        dirs = dirs.copy()
        dirs[flip] *= -1  # eigenvector sign is arbitrary — keep propagation smooth

        stop = (~valid) | (fa_vals < fa_thresh)
        if step > 0:
            prev_norm = np.linalg.norm(prev_dir, axis=-1)
            dir_norm = np.linalg.norm(dirs, axis=-1)
            cos_angle = np.sum(dirs * prev_dir, axis=-1) / (dir_norm * prev_norm + 1e-8)
            cos_angle = np.clip(cos_angle, -1, 1)
            angle = np.degrees(np.arccos(cos_angle))
            stop |= (angle > angle_thresh_deg) & (prev_norm > 0)

        active &= ~stop
        norm = np.linalg.norm(dirs, axis=-1, keepdims=True) + 1e-8
        step_vec = (dirs / norm) * step_size
        pos = np.where(active[:, None], pos + step_vec, pos)
        prev_dir = np.where(active[:, None], dirs, prev_dir)
        all_points.append(pos.copy())
        if not active.any():
            break

    return np.stack(all_points, axis=1)


def _rasterize_streamlines(points, shape):
    mask = np.zeros(shape, dtype=bool)
    idx = np.round(points.reshape(-1, 3)).astype(int)
    valid = np.all((idx >= 0) & (idx < np.array(shape)), axis=-1)
    idx = idx[valid]
    mask[idx[:, 0], idx[:, 1], idx[:, 2]] = True
    return mask


def _dice_score(mask_a, mask_b):
    inter = np.logical_and(mask_a, mask_b).sum()
    total = mask_a.sum() + mask_b.sum()
    return float(2 * inter / total) if total > 0 else None


MAX_TRACT_SEEDS = 400
model.eval()


def _tractography_dice(entry, rng):
    vols = load_subject_volumes(entry)
    if vols is None:
        return None
    t1 = np.asarray(vols["t1"], dtype=np.float32)
    fa_gt = np.asarray(vols["fa"], dtype=np.float32)
    tensor_gt_components = np.asarray(vols["tensor"], dtype=np.float32)
    tensor_gt = lower_tri_to_tensor(tensor_gt_components)  # numpy helper from Step 10
    _, eigvecs_gt = np.linalg.eigh(tensor_gt)
    principal_gt = eigvecs_gt[..., :, -1]

    x = torch.from_numpy(t1[None, None, ...]).to(DEVICE)
    with torch.no_grad():
        pred_log = model(x)
        pred_components = pred_log.permute(0, 2, 3, 4, 1)
        tensor_pred_t = symmetric_expm(lower_tri_to_tensor_torch(pred_components))
    tensor_pred = tensor_pred_t[0].cpu().numpy()
    fa_pred, _ = tensor_fa_md_np(tensor_pred)  # numpy helper from Step 10
    _, eigvecs_pred = np.linalg.eigh(tensor_pred)
    principal_pred = eigvecs_pred[..., :, -1]

    wm_idx = np.argwhere(fa_gt > 0.2)
    if len(wm_idx) == 0:
        return None
    if len(wm_idx) > MAX_TRACT_SEEDS:
        sel = rng.choice(len(wm_idx), size=MAX_TRACT_SEEDS, replace=False)
        seeds = wm_idx[sel].astype(np.float32)
    else:
        seeds = wm_idx.astype(np.float32)

    pts_gt = _track_streamlines(principal_gt, fa_gt, seeds)
    pts_pred = _track_streamlines(principal_pred, fa_pred, seeds)

    mask_gt = _rasterize_streamlines(pts_gt, fa_gt.shape)
    mask_pred = _rasterize_streamlines(pts_pred, fa_pred.shape)

    dice = _dice_score(mask_gt, mask_pred)
    return dict(dice=dice, n_seeds=len(seeds),
                voxels_gt=int(mask_gt.sum()), voxels_pred=int(mask_pred.sum()))


tract_rng = np.random.default_rng(0)
tract_per_subject = []
for entry in test_ds.entries:
    try:
        r = _tractography_dice(entry, tract_rng)
    except Exception as e:
        print(f"  [WARN] tractography failed for {entry['group']}:{entry['subject_id']} -> {e}")
        r = None
    if r is None:
        continue
    r["group"] = entry["group"]
    r["subject_id"] = entry["subject_id"]
    tract_per_subject.append(r)


def _agg_tract(rows):
    if not rows:
        return None
    vals = [r["dice"] for r in rows if r.get("dice") is not None]
    return dict(dice_mean=float(np.mean(vals)) if vals else None,
                dice_std=float(np.std(vals)) if vals else None,
                n_subjects=len(rows))


tractography_results = {"overall": _agg_tract(tract_per_subject)}
for g in ("AD", "MCI", "CN", "HCP"):
    tractography_results[g] = _agg_tract([r for r in tract_per_subject if r["group"] == g])

print(json.dumps(tractography_results, indent=2))
with open("/kaggle/working/tractography_results.json", "w") as f:
    json.dump(tractography_results, f, indent=2)


In [ ]:
import pandas as pd
# ===================== PATCH P4 — add as the LAST cell, just before Step 14 ===============
# Saves everything notebook 04 needs, so all variants are evaluated by ONE piece of code:
#   eval_dump/<VARIANT>/test/<group>__<subject>.npz  (predicted + ground-truth tensor, FA, MD)
#   eval_dump/<VARIANT>/features.csv                 (7 classifier features, real + synthetic, all subjects)
#   eval_dump/<VARIANT>/meta.json
VARIANT = "V1"        # <-- "V1", "V2", "V3" or "V4"
DUMP_DIR = f"/kaggle/working/eval_dump/{VARIANT}"
os.makedirs(os.path.join(DUMP_DIR, "test"), exist_ok=True)

model.load_state_dict(torch.load(BEST_CKPT, map_location=DEVICE)["model_state"])
model.eval()


def _predict(t1):
    x = torch.from_numpy(np.asarray(t1, dtype=np.float32)[None, None]).to(DEVICE)
    with torch.no_grad():
        out = model(x)
        pred_log = out[0] if isinstance(out, (tuple, list)) else out
        T = symmetric_expm(lower_tri_to_tensor_torch(pred_log.float().permute(0, 2, 3, 4, 1)))
        fa_p, md_p = tensor_fa_md(T)
        T6 = tensor_to_lower_tri_torch(T)
    return T6[0].cpu().numpy(), fa_p[0].cpu().numpy(), md_p[0].cpu().numpy()


def _feat7(fa_vol, md_vol):   # identical to Step 12's _extract_features
    wm = fa_vol > 0.2
    return [float(fa_vol.mean()), float(fa_vol.std()), float(md_vol.mean()), float(md_vol.std()),
            float(fa_vol[wm].mean()) if wm.any() else 0.0, float(md_vol[wm].mean()) if wm.any() else 0.0,
            float(wm.mean())]


FEATURE_NAMES_7 = ["fa_mean", "fa_std", "md_mean", "md_std", "wm_fa_mean", "wm_md_mean", "wm_frac"]
feat_rows = []
for split_name, entries in (("train", train_ds.entries), ("val", val_ds.entries), ("test", test_ds.entries)):
    for e in entries:
        vols = load_subject_volumes(e)
        fa_gt = np.asarray(vols["fa"], dtype=np.float32)
        md_gt = np.asarray(vols["md"], dtype=np.float32)
        T6, fa_p, md_p = _predict(vols["t1"])
        if split_name == "test":
            np.savez_compressed(os.path.join(DUMP_DIR, "test", f"{e['group']}__{e['subject_id']}.npz"),
                                pred_tensor=T6.astype(np.float16), gt_tensor=np.asarray(vols["tensor"], dtype=np.float16),
                                fa_gt=fa_gt.astype(np.float16), md_gt=md_gt.astype(np.float16))
        feat_rows.append(dict(group=e["group"], subject_id=e["subject_id"], split=split_name,
                              **{f"real_{k}": v for k, v in zip(FEATURE_NAMES_7, _feat7(fa_gt, md_gt))},
                              **{f"syn_{k}": v for k, v in zip(FEATURE_NAMES_7, _feat7(fa_p, md_p))}))
    print(f"  dumped {split_name}: {len(entries)}")
pd.DataFrame(feat_rows).to_csv(os.path.join(DUMP_DIR, "features.csv"), index=False)
json.dump(dict(variant=VARIANT, best_epoch=int(torch.load(BEST_CKPT, map_location="cpu")["epoch"]),
               affine_harmonization=bool(globals().get("USE_AFFINE_HARMONIZATION", False)),
               ad_factor=AD_FACTOR, split_csv=SPLIT_CSV, n_test=len(test_ds.entries)),
          open(os.path.join(DUMP_DIR, "meta.json"), "w"), indent=1)
for f in ("history.csv", "bidirectional_check_results.json", "tractography_results.json", "test_results.json"):
    if os.path.exists(os.path.join("/kaggle/working", f)):
        shutil.copy(os.path.join("/kaggle/working", f), DUMP_DIR)
print("eval dump ->", DUMP_DIR)

CLEAN_CACHE_AT_END = False   # removes the ~10 GB volume cache from the output so the commit saves faster
if CLEAN_CACHE_AT_END:
    shutil.rmtree(CACHE_DIR, ignore_errors=True)
    print("removed", CACHE_DIR)

## Step 14 - Package outputs for download

In [ ]:
import shutil as _shutil

zip_path = "/kaggle/working/training_outputs"
_shutil.make_archive(zip_path, "zip", "/kaggle/working", "checkpoints")
# also fold in the loose files (history, results, plots) — re-zip everything together
staging = "/kaggle/working/_package"
os.makedirs(staging, exist_ok=True)
for fname in ["history.csv", "test_results.json", "test_results_per_subject.json",
              "bidirectional_check_results.json", "diagnostic_validity_results.json",
              "tractography_results.json",
              "training_curves.png", "qualitative_check.png", "color_fa_check.png"]:
    src = os.path.join("/kaggle/working", fname)
    if os.path.exists(src):
        _shutil.copy(src, staging)
if os.path.exists(os.path.join("/kaggle/working", "checkpoints", "best.pt")):
    os.makedirs(os.path.join(staging, "checkpoints"), exist_ok=True)
    _shutil.copy(os.path.join("/kaggle/working", "checkpoints", "best.pt"),
                 os.path.join(staging, "checkpoints", "best.pt"))

final_zip = _shutil.make_archive("/kaggle/working/final_package", "zip", staging)
size_mb = os.path.getsize(final_zip) / (1024 * 1024)
print(f"Packaged: {final_zip} ({size_mb:.1f} MB)")
print("Download it from the Output pane on the right, or click 'Save Version' to persist everything.")
